In [8]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

INPUT_DIR = "../data/calibration_set"
TYPES = ["correctness", "efficiency", "syntax"]

# Load human annotations
annotated = pd.concat([
    pd.read_json(f"{INPUT_DIR}/annotated_subset_1.jsonl", lines=True),
    pd.read_json(f"{INPUT_DIR}/annotated_subset_2.jsonl", lines=True)
], ignore_index=True)

for metric in TYPES:
    score_col = f"{metric}_score"

    llm = pd.read_json(
        f"{INPUT_DIR}/{metric}.jsonl",
        lines=True
    )

    merged = pd.merge(
        annotated[["sub_id", score_col]],
        llm[["sub_id", score_col]],
        on="sub_id",
        suffixes=("_human", "_llm")
    )

    kappa = cohen_kappa_score(
        merged[f"{score_col}_human"],
        merged[f"{score_col}_llm"],
        weights="quadratic"
    )

    print(f"{metric:<12}: {kappa:.3f}")

correctness : 0.940
efficiency  : 0.909
syntax      : 0.904
